In [ ]:
from adaptive_latents import datasets

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from types import SimpleNamespace
from pathlib import Path
import pims

from PIL import Image


In [ ]:
datasets.Zong22Dataset.sub_datset_info

In [ ]:
d1 = datasets.Zong22Dataset(11)
d2 = datasets.Zong22Dataset(12)
plt.plot(d1.neural_data.t, d1.neural_data)
plt.plot(d2.neural_data.t + d1.neural_data.t[-1] + d1.neural_data.dt * 1, d2.neural_data)

plt.xlim(np.array([-1,2]) * d1.neural_data.dt + d1.neural_data.t[-1])
plt.ylim(-.5,2)


In [ ]:
import tifffile
file = d1.dataset_base_path/d1.sdi_row.basepath/ d1.sdi_row.raw_frames
t = tifffile.TiffFile(file)
a = tifffile.imread(file)

In [ ]:
d1.neural_data.shape

In [ ]:
d1.neural_data.dt

In [ ]:
a.shape

In [ ]:
d1.neural_data.shape

In [ ]:
d1.F_all.shape[1] / 8700

In [ ]:
d1.n_cells

In [ ]:
d1.neural_data.shape[0] / 17400

In [ ]:
self = datasets.Zong22Dataset()

def make_open_field_entry(area, animal_id, date, f_part, f_total, recording_date=None):
    recording_date = date if recording_date is None else recording_date
    sub_dataset_info = {
        'basepath': f'{area}_recordings/{animal_id}/{date}/',
        'part_of_F': (f_part,f_total),
        'raw_frames':     f'{animal_id}_{recording_date}_ML-400_AL-400_1Openfiled_0000{f_part}.tif',
        'behavior_csv': f'{animal_id}_{recording_date}_ML-400_AL-400_1Openfiled_0000{f_part}_trackingVideoDLC_resnet_50_OPENMINI2P_topcamera_20210305Mar5shuffle1_1030000.csv',
    }
    return sub_dataset_info


self.sub_datset_info.loc[8] = make_open_field_entry(area='MEC', animal_id='97045', date='20210305', f_part=1, f_total=6, recording_date='20210304')
self.sub_dataset = 8



In [ ]:
self.dataset_base_path

In [ ]:
self.sub_datset_info

In [ ]:
self.dataset_base_path / self.sub_datset_info.basepath[self.sub_dataset]

In [ ]:
sub_dataset_base_path = self.dataset_base_path / self.sub_datset_info.basepath[self.sub_dataset]
if not sub_dataset_base_path.is_dir():
    print(f"Go download the dataset from {self.doi}. (Or remount the external drive on Tycho)")
    raise FileNotFoundError()

iscell = np.load(sub_dataset_base_path / 'suite2p' / 'plane0' / 'iscell.npy')
F_all = np.load(sub_dataset_base_path / 'suite2p' / 'plane0' / 'F.npy')
self.F_all = F_all
n_cells = int(sum(iscell[:, 0]))

stat = np.load(sub_dataset_base_path / 'suite2p' / 'plane0' / 'stat.npy', allow_pickle=True)
ops = np.load(sub_dataset_base_path / 'suite2p' / 'plane0' / 'ops.npy', allow_pickle=True).item()

def make_beh(fpath):
    pre_beh = pd.read_csv(fpath)
    columns = ["t"] + list(map(lambda a: f"{a[0]}_{a[1]}", zip(pre_beh.iloc[0, 1:], pre_beh.iloc[1, 1:])))
    columns = {pre_beh.columns[i]: columns[i] for i in range(len(columns))}
    beh = pre_beh.rename(columns=columns).iloc[2:].astype(float).reset_index(drop=True)
    beh.t = beh.t / self.neural_Fs
    return beh

part, total = self.sub_datset_info.part_of_F[self.sub_dataset]
block_length = F_all.shape[1] // total

F_all = F_all - F_all.min(axis=1, keepdims=True)
# F_all = F_all / np.median(F_all, axis=1, keepdims=True)

F_all_0 = np.median(F_all, axis=1, keepdims=True)
F_all = (F_all - F_all_0) / F_all_0

F_all[np.isnan(F_all)] = 0

F = F_all[:, (part - 1) * block_length: part * block_length]
img = Image.open(sub_dataset_base_path / self.sub_datset_info.raw_frames[self.sub_dataset])
video = None
if isinstance(video_filename:=self.sub_datset_info.behavior_video[self.sub_dataset], str):
    video = pims.Video(sub_dataset_base_path / video_filename)
beh = make_beh(sub_dataset_base_path / self.sub_datset_info.behavior_csv[self.sub_dataset])


if 'nose_x' in beh:
    nose = self.get_behavior_trace(beh, 'nose')
body = self.get_behavior_trace(beh, 'bodycenter')

head = (self.get_behavior_trace(beh, 'leftear') + self.get_behavior_trace(beh, 'rightear'))/2


if 'nose_x' in beh:
    beh['hd'] = np.arctan2(*(nose - head).T)
beh['h2b'] = np.linalg.norm(head - body, axis=1)
beh['x'] = head[:,0]
beh['y'] = head[:,1]


# return F, img, video, beh, n_cells, stat, ops


In [ ]:
plt.imshow(F)

In [ ]:
'leftear_x' in beh

In [ ]:

from adaptive_latents import datasets

In [ ]:
d = datasets.Zong22Dataset(7)

In [ ]:
d.cells_per_pane

In [ ]:
import h5py

f = h5py.File(d.dataset_base_path / d.sdi_row.basepath / 'GridCellAnalysis.mat')

In [ ]:
f[f['GridCellAnalysis']['ActivityMap'][0,0]]

In [ ]:
f[f['GridCellAnalysis']['IsGridCell'][0,0]][:]